# Demographic Analysis of Medical Error Detection Results

This notebook analyzes prediction results across different demographic groups, disease types, and medical departments.

## Analysis Dimensions
- **Demographics**: Age, Gender, Age Groups
- **Error Analysis**: Error Type, Prediction Accuracy
- **Debate Analysis**: Winner (Expert A vs B)
- **Disease Type**: Infectious, Cardiovascular, Respiratory, etc.
- **Department**: Emergency, Internal Medicine, Pediatrics, etc.

In [ ]:
# Import libraries
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc
)

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Configuration

In [ ]:
# File paths - adjust these as needed
VALIDATION_FILE = "../test_data/validation.json"
RESULTS_DIR = "../logs/debates"
OUTPUT_DIR = "./analysis"

# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Validation file: {VALIDATION_FILE}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Define Extraction Functions

In [ ]:
# Age group definitions
AGE_GROUPS = {
    'Pediatric (0-17)': (0, 17),
    'Young Adult (18-35)': (18, 35),
    'Middle-aged (36-55)': (36, 55),
    'Older Adult (56+)': (56, 200)
}

def extract_age(text):
    """Extract age from medical note text."""
    patterns = [
        r'(\d{1,3})[-\s]?year[-\s]?old',
        r'aged?\s*(\d{1,3})',
        r'(\d{1,3})\s*years?\s*old',
        r'(\d{1,3})\s*yo\b',
        r'(\d{1,3})\s*y/?o\b',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            age = int(match.group(1))
            if 0 <= age <= 120:
                return age
    return None

def extract_gender(text):
    """Extract gender from medical note text."""
    text_lower = text.lower()
    
    # Check for explicit patterns
    if re.search(r'\d+[-\s]?year[-\s]?old\s+(woman|female|girl)', text_lower):
        return 'Female'
    if re.search(r'\d+[-\s]?year[-\s]?old\s+(man|male|boy)', text_lower):
        return 'Male'
    
    # Count gender indicators
    male_count = len(re.findall(r'\b(man|male|boy|he|his|him)\b', text_lower))
    female_count = len(re.findall(r'\b(woman|female|girl|she|her)\b', text_lower))
    
    if female_count > male_count:
        return 'Female'
    elif male_count > female_count:
        return 'Male'
    return 'Unknown'

def get_age_group(age):
    """Map age to age group."""
    if age is None:
        return 'Unknown'
    for group_name, (min_age, max_age) in AGE_GROUPS.items():
        if min_age <= age <= max_age:
            return group_name
    return 'Unknown'

In [ ]:
# Disease type classification based on keywords
DISEASE_KEYWORDS = {
    'Infectious Disease': [
        'infection', 'bacterial', 'viral', 'virus', 'bacteria', 'sepsis', 'septic',
        'pneumonia', 'tuberculosis', 'HIV', 'AIDS', 'hepatitis', 'meningitis',
        'gonorrhea', 'chlamydia', 'syphilis', 'herpes', 'influenza', 'flu',
        'streptococcus', 'staphylococcus', 'e. coli', 'salmonella', 'malaria',
        'typhoid', 'cholera', 'UTI', 'urinary tract infection', 'cellulitis',
        'abscess', 'fever', 'antimicrobial', 'antibiotic', 'pathogen', 'organism'
    ],
    'Cardiovascular': [
        'heart', 'cardiac', 'cardiovascular', 'myocardial', 'infarction',
        'arrhythmia', 'atrial', 'ventricular', 'hypertension', 'hypotension',
        'angina', 'coronary', 'aortic', 'valve', 'murmur', 'palpitation',
        'tachycardia', 'bradycardia', 'chest pain', 'EKG', 'ECG', 'echocardiogram'
    ],
    'Respiratory': [
        'lung', 'pulmonary', 'respiratory', 'bronchitis', 'asthma', 'COPD',
        'emphysema', 'cough', 'dyspnea', 'shortness of breath', 'wheeze',
        'stridor', 'pneumothorax', 'pleural', 'bronchial', 'alveolar'
    ],
    'Gastrointestinal': [
        'stomach', 'intestinal', 'bowel', 'colon', 'gastric', 'hepatic', 'liver',
        'pancreatic', 'pancreatitis', 'appendicitis', 'cholecystitis', 'diarrhea',
        'vomiting', 'nausea', 'abdominal pain', 'GI', 'gastrointestinal'
    ],
    'Neurological': [
        'brain', 'neurological', 'seizure', 'epilepsy', 'stroke', 'TIA',
        'headache', 'migraine', 'meningitis', 'encephalitis', 'neuropathy',
        'paralysis', 'weakness', 'numbness', 'tingling', 'consciousness'
    ],
    'Musculoskeletal': [
        'bone', 'joint', 'muscle', 'arthritis', 'fracture', 'osteoporosis',
        'back pain', 'knee', 'hip', 'shoulder', 'spine', 'orthopedic',
        'arthroplasty', 'prosthesis'
    ],
    'Dermatological': [
        'skin', 'rash', 'lesion', 'dermatitis', 'eczema', 'psoriasis',
        'wound', 'ulcer', 'blister', 'erythema', 'pruritus', 'itching'
    ],
    'Endocrine/Metabolic': [
        'diabetes', 'diabetic', 'thyroid', 'insulin', 'glucose', 'metabolic',
        'hormone', 'endocrine', 'obesity', 'hypoglycemia', 'hyperglycemia'
    ],
    'Genitourinary': [
        'kidney', 'renal', 'bladder', 'urinary', 'prostate', 'testicular',
        'ovarian', 'uterine', 'vaginal', 'genital', 'genitourinary', 'STI', 'STD'
    ],
    'Oncology': [
        'cancer', 'tumor', 'malignant', 'benign', 'carcinoma', 'sarcoma',
        'lymphoma', 'leukemia', 'metastasis', 'oncology', 'chemotherapy'
    ],
    'Hematological': [
        'blood', 'anemia', 'leukocyte', 'hemoglobin', 'platelet', 'coagulation',
        'bleeding', 'clot', 'thrombosis', 'hematology'
    ],
    'Immunological': [
        'immune', 'autoimmune', 'allergy', 'allergic', 'immunodeficiency',
        'HIV', 'AIDS', 'lupus', 'rheumatoid'
    ]
}

def classify_disease_type(text):
    """Classify disease type based on keywords in medical note."""
    text_lower = text.lower()
    scores = {}
    
    for disease_type, keywords in DISEASE_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw.lower() in text_lower)
        if score > 0:
            scores[disease_type] = score
    
    if scores:
        # Return the disease type with highest score
        return max(scores, key=scores.get)
    return 'Other/Unclassified'

In [ ]:
# Department classification
DEPARTMENT_KEYWORDS = {
    'Emergency Medicine': [
        'emergency department', 'emergency room', 'ER', 'ED', 'trauma',
        'acute', 'urgent', 'brought to', 'comes to the emergency'
    ],
    'Internal Medicine': [
        'internist', 'internal medicine', 'general medicine', 'primary care',
        'comes to the physician', 'comes to the doctor', 'follow-up', 'outpatient'
    ],
    'Pediatrics': [
        'pediatric', 'pediatrician', 'child', 'infant', 'newborn', 'neonate',
        'toddler', 'adolescent', 'year-old boy', 'year-old girl', 'brought by mother',
        'brought by father', 'brought by parent'
    ],
    'Obstetrics/Gynecology': [
        'obstetric', 'gynecology', 'OB/GYN', 'pregnant', 'pregnancy', 'prenatal',
        'postpartum', 'menstrual', 'vaginal', 'cervical', 'uterine', 'ovarian'
    ],
    'Surgery': [
        'surgery', 'surgical', 'operation', 'post-operative', 'pre-operative',
        'incision', 'resection', 'removal', 'arthroplasty', 'appendectomy'
    ],
    'Cardiology': [
        'cardiology', 'cardiologist', 'heart', 'cardiac', 'EKG', 'echocardiogram',
        'chest pain', 'palpitation', 'arrhythmia'
    ],
    'Pulmonology': [
        'pulmonology', 'pulmonologist', 'lung', 'respiratory', 'breathing',
        'asthma', 'COPD', 'bronchitis'
    ],
    'Infectious Disease': [
        'infectious disease', 'infection', 'sepsis', 'HIV', 'AIDS',
        'antimicrobial', 'antibiotic therapy'
    ],
    'Dermatology': [
        'dermatology', 'dermatologist', 'skin', 'rash', 'lesion', 'dermatitis'
    ],
    'Neurology': [
        'neurology', 'neurologist', 'seizure', 'stroke', 'headache', 'numbness'
    ],
    'Orthopedics': [
        'orthopedic', 'orthopedist', 'fracture', 'joint', 'bone', 'arthroplasty'
    ],
    'Urology': [
        'urology', 'urologist', 'kidney', 'bladder', 'urinary', 'prostate'
    ]
}

def classify_department(text, age=None):
    """Classify department based on keywords and patient age."""
    text_lower = text.lower()
    scores = {}
    
    # Check for pediatric age first
    if age is not None and age < 18:
        scores['Pediatrics'] = 10  # High base score for pediatric patients
    
    for dept, keywords in DEPARTMENT_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw.lower() in text_lower)
        if score > 0:
            scores[dept] = scores.get(dept, 0) + score
    
    if scores:
        return max(scores, key=scores.get)
    return 'General Medicine'

## 3. Load Data

In [ ]:
# Load validation data
with open(VALIDATION_FILE, 'r', encoding='utf-8') as f:
    validation_data = json.load(f)

validation_dict = {case['id']: case for case in validation_data}
print(f"Loaded {len(validation_dict)} validation cases")

In [ ]:
# Load prediction results
results_dir = Path(RESULTS_DIR)
result_files = list(results_dir.glob("result_*.json"))

results_data = {}
for filepath in result_files:
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            result = json.load(f)
        case_id = result.get('case_id')
        if case_id:
            results_data[case_id] = result
    except Exception as e:
        print(f"Error loading {filepath}: {e}")

print(f"Loaded {len(results_data)} prediction results")

## 4. Merge and Extract Features

In [ ]:
# Merge data and extract all features
merged_records = []

for case_id, result in results_data.items():
    val_case = validation_dict.get(case_id, {})
    text = val_case.get('text', result.get('medical_note', ''))
    
    # Extract demographics
    age = extract_age(text)
    gender = extract_gender(text)
    age_group = get_age_group(age)
    
    # Extract disease type and department
    disease_type = classify_disease_type(text)
    department = classify_department(text, age)
    
    # Build record
    record = {
        'case_id': case_id,
        
        # Demographics
        'age': age,
        'age_group': age_group,
        'gender': gender,
        
        # Disease & Department
        'disease_type': disease_type,
        'department': department,
        
        # Ground truth
        'ground_truth': result.get('ground_truth', val_case.get('label')),
        'ground_truth_label': 'INCORRECT' if result.get('ground_truth', val_case.get('label')) == 1 else 'CORRECT',
        'error_type': result.get('error_type', val_case.get('error_type', 'NA')),
        
        # Predictions
        'predicted_label': result.get('predicted_label'),
        'predicted_label_str': 'INCORRECT' if result.get('predicted_label') == 1 else 'CORRECT',
        'final_answer': result.get('final_answer'),
        'confidence_score': result.get('confidence_score'),
        'confidence_normalized': result.get('confidence_normalized'),
        
        # Debate info
        'winner': result.get('winner'),
        
        # Correctness
        'is_correct': result.get('ground_truth') == result.get('predicted_label'),
        
        # Metadata
        'execution_time': result.get('execution_time'),
        'text_length': len(text)
    }
    
    merged_records.append(record)

# Create DataFrame
df = pd.DataFrame(merged_records)
print(f"Created DataFrame with {len(df)} records and {len(df.columns)} columns")
df.head()

In [ ]:
# Data overview
print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"\nTotal cases: {len(df)}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

## 5. Overall Performance Metrics

In [ ]:
# Calculate overall metrics
y_true = df['ground_truth'].values
y_pred = df['predicted_label'].values

# Calculate probability scores for ROC-AUC
# If predicted INCORRECT (1), use confidence as probability of INCORRECT
# If predicted CORRECT (0), use 1-confidence as probability of INCORRECT
y_scores = []
for _, row in df.iterrows():
    if row['predicted_label'] == 1:
        y_scores.append(row['confidence_normalized'] if row['confidence_normalized'] is not None else 0.5)
    else:
        y_scores.append(1 - row['confidence_normalized'] if row['confidence_normalized'] is not None else 0.5)
y_scores = np.array(y_scores)

print("=" * 60)
print("OVERALL PERFORMANCE")
print("=" * 60)

print(f"\nAccuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

# Calculate ROC-AUC
try:
    roc_auc = roc_auc_score(y_true, y_scores)
    print(f"ROC-AUC:   {roc_auc:.4f}")
except Exception as e:
    print(f"ROC-AUC:   Could not calculate ({e})")
    roc_auc = None

print("\n" + "-" * 40)
print("Classification Report:")
print("-" * 40)
print(classification_report(y_true, y_pred, target_names=['CORRECT (0)', 'INCORRECT (1)']))

In [ ]:
# Confusion Matrix and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
ax1 = axes[0]
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['CORRECT', 'INCORRECT'],
            yticklabels=['CORRECT', 'INCORRECT'],
            ax=ax1)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
ax1.set_title('Confusion Matrix')

# ROC Curve
ax2 = axes[1]
if roc_auc is not None:
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random (AUC = 0.5)')
    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title('ROC Curve')
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'ROC curve not available', ha='center', va='center', fontsize=12)
    ax2.set_title('ROC Curve')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix_and_roc.png", dpi=150)
plt.show()

## 6. Demographic Analysis

In [ ]:
def calculate_group_metrics(group_df):
    """Calculate metrics for a group including ROC-AUC."""
    if len(group_df) == 0:
        return {'count': 0, 'accuracy': None, 'precision': None, 'recall': None, 'f1': None, 'roc_auc': None}
    
    y_true = group_df['ground_truth'].values
    y_pred = group_df['predicted_label'].values
    
    metrics = {
        'count': len(group_df),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'avg_confidence': group_df['confidence_normalized'].mean()
    }
    
    # Calculate ROC-AUC using confidence scores
    try:
        if len(set(y_true)) > 1:  # Need both classes for ROC-AUC
            y_scores = []
            for _, row in group_df.iterrows():
                if row['predicted_label'] == 1:
                    y_scores.append(row['confidence_normalized'] if row['confidence_normalized'] is not None else 0.5)
                else:
                    y_scores.append(1 - row['confidence_normalized'] if row['confidence_normalized'] is not None else 0.5)
            metrics['roc_auc'] = roc_auc_score(y_true, y_scores)
        else:
            metrics['roc_auc'] = None
    except Exception:
        metrics['roc_auc'] = None
    
    return metrics

### 6.1 Age Distribution

In [ ]:
# Age distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of ages
ax1 = axes[0]
df['age'].dropna().hist(bins=20, ax=ax1, color='steelblue', edgecolor='black')
ax1.set_xlabel('Age')
ax1.set_ylabel('Count')
ax1.set_title('Age Distribution')

# Age group distribution
ax2 = axes[1]
age_group_order = ['Pediatric (0-17)', 'Young Adult (18-35)', 'Middle-aged (36-55)', 'Older Adult (56+)', 'Unknown']
age_counts = df['age_group'].value_counts().reindex(age_group_order).dropna()
age_counts.plot(kind='bar', ax=ax2, color='steelblue', edgecolor='black')
ax2.set_xlabel('Age Group')
ax2.set_ylabel('Count')
ax2.set_title('Age Group Distribution')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/age_distribution.png", dpi=150)
plt.show()

In [ ]:
# Performance by Age Group
print("\n" + "=" * 60)
print("PERFORMANCE BY AGE GROUP")
print("=" * 60)

age_metrics = []
for age_group in age_group_order:
    group_df = df[df['age_group'] == age_group]
    if len(group_df) > 0:
        metrics = calculate_group_metrics(group_df)
        metrics['age_group'] = age_group
        age_metrics.append(metrics)

age_metrics_df = pd.DataFrame(age_metrics)
age_metrics_df = age_metrics_df[['age_group', 'count', 'accuracy', 'precision', 'recall', 'f1', 'avg_confidence']]
print(age_metrics_df.to_string(index=False))

In [ ]:
# Accuracy by Age Group visualization
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(age_metrics_df))
bars = ax.bar(x, age_metrics_df['accuracy'], color='steelblue', edgecolor='black')

# Add count labels on bars
for i, (bar, count) in enumerate(zip(bars, age_metrics_df['count'])):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'n={count}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(age_metrics_df['age_group'], rotation=45, ha='right')
ax.set_xlabel('Age Group')
ax.set_ylabel('Accuracy')
ax.set_title('Prediction Accuracy by Age Group')
ax.set_ylim(0, 1.1)
ax.axhline(y=df['is_correct'].mean(), color='red', linestyle='--', label='Overall Accuracy')
ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/accuracy_by_age_group.png", dpi=150)
plt.show()

### 6.2 Gender Analysis

In [ ]:
# Gender distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
ax1 = axes[0]
gender_counts = df['gender'].value_counts()
colors = ['#ff9999', '#66b3ff', '#99ff99']
ax1.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Gender Distribution')

# Accuracy by gender
ax2 = axes[1]
gender_accuracy = df.groupby('gender')['is_correct'].mean()
gender_counts_for_plot = df.groupby('gender').size()
bars = ax2.bar(gender_accuracy.index, gender_accuracy.values, color=['#ff9999', '#66b3ff', '#99ff99'], edgecolor='black')

for bar, count in zip(bars, gender_counts_for_plot):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'n={count}', ha='center', va='bottom')

ax2.set_xlabel('Gender')
ax2.set_ylabel('Accuracy')
ax2.set_title('Prediction Accuracy by Gender')
ax2.set_ylim(0, 1.1)
ax2.axhline(y=df['is_correct'].mean(), color='red', linestyle='--', label='Overall')
ax2.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/gender_analysis.png", dpi=150)
plt.show()

In [ ]:
# Performance by Gender
print("\n" + "=" * 60)
print("PERFORMANCE BY GENDER")
print("=" * 60)

gender_metrics = []
for gender in ['Female', 'Male', 'Unknown']:
    group_df = df[df['gender'] == gender]
    if len(group_df) > 0:
        metrics = calculate_group_metrics(group_df)
        metrics['gender'] = gender
        gender_metrics.append(metrics)

gender_metrics_df = pd.DataFrame(gender_metrics)
gender_metrics_df = gender_metrics_df[['gender', 'count', 'accuracy', 'precision', 'recall', 'f1', 'avg_confidence']]
print(gender_metrics_df.to_string(index=False))

## 7. Error Type Analysis

In [ ]:
# Error type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
ax1 = axes[0]
error_counts = df['error_type'].value_counts()
error_counts.plot(kind='bar', ax=ax1, color='coral', edgecolor='black')
ax1.set_xlabel('Error Type')
ax1.set_ylabel('Count')
ax1.set_title('Error Type Distribution')
ax1.tick_params(axis='x', rotation=45)

# Accuracy by error type
ax2 = axes[1]
error_accuracy = df.groupby('error_type')['is_correct'].mean().sort_values(ascending=False)
error_counts_plot = df.groupby('error_type').size().reindex(error_accuracy.index)
bars = ax2.bar(error_accuracy.index, error_accuracy.values, color='coral', edgecolor='black')

for bar, count in zip(bars, error_counts_plot):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'n={count}', ha='center', va='bottom', fontsize=9)

ax2.set_xlabel('Error Type')
ax2.set_ylabel('Accuracy')
ax2.set_title('Prediction Accuracy by Error Type')
ax2.tick_params(axis='x', rotation=45)
ax2.set_ylim(0, 1.1)
ax2.axhline(y=df['is_correct'].mean(), color='red', linestyle='--', label='Overall')
ax2.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/error_type_analysis.png", dpi=150)
plt.show()

In [ ]:
# Performance by Error Type
print("\n" + "=" * 60)
print("PERFORMANCE BY ERROR TYPE")
print("=" * 60)

error_metrics = []
for error_type in df['error_type'].unique():
    group_df = df[df['error_type'] == error_type]
    metrics = calculate_group_metrics(group_df)
    metrics['error_type'] = error_type
    error_metrics.append(metrics)

error_metrics_df = pd.DataFrame(error_metrics)
error_metrics_df = error_metrics_df.sort_values('count', ascending=False)
error_metrics_df = error_metrics_df[['error_type', 'count', 'accuracy', 'precision', 'recall', 'f1', 'avg_confidence']]
print(error_metrics_df.to_string(index=False))

## 8. Disease Type Analysis

In [ ]:
# Disease type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution
ax1 = axes[0]
disease_counts = df['disease_type'].value_counts()
disease_counts.plot(kind='barh', ax=ax1, color='mediumseagreen', edgecolor='black')
ax1.set_xlabel('Count')
ax1.set_ylabel('Disease Type')
ax1.set_title('Disease Type Distribution')

# Accuracy by disease type
ax2 = axes[1]
disease_accuracy = df.groupby('disease_type')['is_correct'].mean().sort_values(ascending=True)
disease_counts_plot = df.groupby('disease_type').size().reindex(disease_accuracy.index)
bars = ax2.barh(disease_accuracy.index, disease_accuracy.values, color='mediumseagreen', edgecolor='black')

for bar, count in zip(bars, disease_counts_plot):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
             f'n={count}', ha='left', va='center', fontsize=9)

ax2.set_xlabel('Accuracy')
ax2.set_ylabel('Disease Type')
ax2.set_title('Prediction Accuracy by Disease Type')
ax2.set_xlim(0, 1.2)
ax2.axvline(x=df['is_correct'].mean(), color='red', linestyle='--', label='Overall')
ax2.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/disease_type_analysis.png", dpi=150)
plt.show()

In [ ]:
# Performance by Disease Type
print("\n" + "=" * 60)
print("PERFORMANCE BY DISEASE TYPE")
print("=" * 60)

disease_metrics = []
for disease_type in df['disease_type'].unique():
    group_df = df[df['disease_type'] == disease_type]
    metrics = calculate_group_metrics(group_df)
    metrics['disease_type'] = disease_type
    disease_metrics.append(metrics)

disease_metrics_df = pd.DataFrame(disease_metrics)
disease_metrics_df = disease_metrics_df.sort_values('count', ascending=False)
disease_metrics_df = disease_metrics_df[['disease_type', 'count', 'accuracy', 'precision', 'recall', 'f1', 'avg_confidence']]
print(disease_metrics_df.to_string(index=False))

## 9. Department Analysis

In [ ]:
# Department distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution
ax1 = axes[0]
dept_counts = df['department'].value_counts()
dept_counts.plot(kind='barh', ax=ax1, color='mediumpurple', edgecolor='black')
ax1.set_xlabel('Count')
ax1.set_ylabel('Department')
ax1.set_title('Department Distribution')

# Accuracy by department
ax2 = axes[1]
dept_accuracy = df.groupby('department')['is_correct'].mean().sort_values(ascending=True)
dept_counts_plot = df.groupby('department').size().reindex(dept_accuracy.index)
bars = ax2.barh(dept_accuracy.index, dept_accuracy.values, color='mediumpurple', edgecolor='black')

for bar, count in zip(bars, dept_counts_plot):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
             f'n={count}', ha='left', va='center', fontsize=9)

ax2.set_xlabel('Accuracy')
ax2.set_ylabel('Department')
ax2.set_title('Prediction Accuracy by Department')
ax2.set_xlim(0, 1.2)
ax2.axvline(x=df['is_correct'].mean(), color='red', linestyle='--', label='Overall')
ax2.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/department_analysis.png", dpi=150)
plt.show()

In [ ]:
# Performance by Department
print("\n" + "=" * 60)
print("PERFORMANCE BY DEPARTMENT")
print("=" * 60)

dept_metrics = []
for dept in df['department'].unique():
    group_df = df[df['department'] == dept]
    metrics = calculate_group_metrics(group_df)
    metrics['department'] = dept
    dept_metrics.append(metrics)

dept_metrics_df = pd.DataFrame(dept_metrics)
dept_metrics_df = dept_metrics_df.sort_values('count', ascending=False)
dept_metrics_df = dept_metrics_df[['department', 'count', 'accuracy', 'precision', 'recall', 'f1', 'avg_confidence']]
print(dept_metrics_df.to_string(index=False))

## 10. Winner Analysis (Expert A vs Expert B)

In [ ]:
# Winner distribution and accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribution
ax1 = axes[0]
winner_counts = df['winner'].value_counts()
colors_winner = ['#3498db', '#e74c3c', '#95a5a6']
winner_counts.plot(kind='bar', ax=ax1, color=colors_winner, edgecolor='black')
ax1.set_xlabel('Winner')
ax1.set_ylabel('Count')
ax1.set_title('Debate Winner Distribution')
ax1.tick_params(axis='x', rotation=0)

# Accuracy by winner
ax2 = axes[1]
winner_accuracy = df.groupby('winner')['is_correct'].mean()
winner_counts_plot = df.groupby('winner').size()
bars = ax2.bar(winner_accuracy.index, winner_accuracy.values, color=colors_winner, edgecolor='black')

for bar, count in zip(bars, winner_counts_plot):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'n={count}', ha='center', va='bottom')

ax2.set_xlabel('Winner')
ax2.set_ylabel('Accuracy')
ax2.set_title('Prediction Accuracy by Debate Winner')
ax2.set_ylim(0, 1.1)
ax2.axhline(y=df['is_correct'].mean(), color='red', linestyle='--', label='Overall')
ax2.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/winner_analysis.png", dpi=150)
plt.show()

In [ ]:
# Performance by Winner
print("\n" + "=" * 60)
print("PERFORMANCE BY DEBATE WINNER")
print("=" * 60)

winner_metrics = []
for winner in df['winner'].dropna().unique():
    group_df = df[df['winner'] == winner]
    metrics = calculate_group_metrics(group_df)
    metrics['winner'] = winner
    winner_metrics.append(metrics)

winner_metrics_df = pd.DataFrame(winner_metrics)
winner_metrics_df = winner_metrics_df[['winner', 'count', 'accuracy', 'precision', 'recall', 'f1', 'avg_confidence']]
print(winner_metrics_df.to_string(index=False))

## 11. Cross-tabulation Analysis

In [ ]:
# Age Group vs Error Type Heatmap
crosstab_age_error = pd.crosstab(df['age_group'], df['error_type'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(crosstab_age_error, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Age Group vs Error Type')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/crosstab_age_error.png", dpi=150)
plt.show()

In [ ]:
# Disease Type vs Department Heatmap
crosstab_disease_dept = pd.crosstab(df['disease_type'], df['department'])

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(crosstab_disease_dept, annot=True, fmt='d', cmap='YlGnBu', ax=ax)
ax.set_title('Disease Type vs Department')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/crosstab_disease_dept.png", dpi=150)
plt.show()

In [ ]:
# Gender vs Accuracy by Age Group
pivot_gender_age = df.pivot_table(
    values='is_correct', 
    index='gender', 
    columns='age_group', 
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_gender_age, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, vmin=0, vmax=1)
ax.set_title('Accuracy: Gender vs Age Group')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/crosstab_gender_age_accuracy.png", dpi=150)
plt.show()

## 12. Confidence Analysis

In [ ]:
# Confidence distribution by correctness
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
ax1 = axes[0]
df[df['is_correct'] == True]['confidence_normalized'].hist(bins=10, ax=ax1, alpha=0.7, label='Correct', color='green')
df[df['is_correct'] == False]['confidence_normalized'].hist(bins=10, ax=ax1, alpha=0.7, label='Incorrect', color='red')
ax1.set_xlabel('Confidence Score (Normalized)')
ax1.set_ylabel('Count')
ax1.set_title('Confidence Distribution by Prediction Correctness')
ax1.legend()

# Box plot
ax2 = axes[1]
df.boxplot(column='confidence_normalized', by='is_correct', ax=ax2)
ax2.set_xlabel('Prediction Correct')
ax2.set_ylabel('Confidence Score')
ax2.set_title('Confidence by Correctness')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confidence_analysis.png", dpi=150)
plt.show()

In [ ]:
# Confidence calibration
print("\n" + "=" * 60)
print("CONFIDENCE CALIBRATION")
print("=" * 60)

# Bin confidence scores
df['confidence_bin'] = pd.cut(df['confidence_normalized'], bins=[0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
calibration = df.groupby('confidence_bin').agg({
    'is_correct': ['mean', 'count'],
    'confidence_normalized': 'mean'
}).round(3)
calibration.columns = ['accuracy', 'count', 'avg_confidence']
print(calibration)

## 13. Summary Dashboard

In [ ]:
# Create summary dashboard
fig = plt.figure(figsize=(16, 12))

# Overall metrics text
ax1 = fig.add_subplot(2, 3, 1)
ax1.axis('off')
roc_auc_str = f"{roc_auc:.4f}" if roc_auc is not None else "N/A"
overall_text = f"""
OVERALL METRICS
{'='*30}
Total Cases: {len(df)}
Accuracy:    {accuracy_score(y_true, y_pred):.4f}
Precision:   {precision_score(y_true, y_pred, zero_division=0):.4f}
Recall:      {recall_score(y_true, y_pred, zero_division=0):.4f}
F1 Score:    {f1_score(y_true, y_pred, zero_division=0):.4f}
ROC-AUC:     {roc_auc_str}
"""
ax1.text(0.1, 0.5, overall_text, fontsize=12, family='monospace', verticalalignment='center')

# Age group accuracy
ax2 = fig.add_subplot(2, 3, 2)
age_acc = df.groupby('age_group')['is_correct'].mean()
age_acc.plot(kind='bar', ax=ax2, color='steelblue', edgecolor='black')
ax2.set_title('Accuracy by Age Group')
ax2.set_ylim(0, 1)
ax2.tick_params(axis='x', rotation=45)

# Gender accuracy
ax3 = fig.add_subplot(2, 3, 3)
gender_acc = df.groupby('gender')['is_correct'].mean()
gender_acc.plot(kind='bar', ax=ax3, color=['#ff9999', '#66b3ff', '#99ff99'], edgecolor='black')
ax3.set_title('Accuracy by Gender')
ax3.set_ylim(0, 1)

# Disease type accuracy
ax4 = fig.add_subplot(2, 3, 4)
disease_acc = df.groupby('disease_type')['is_correct'].mean().sort_values(ascending=True)
disease_acc.plot(kind='barh', ax=ax4, color='mediumseagreen', edgecolor='black')
ax4.set_title('Accuracy by Disease Type')
ax4.set_xlim(0, 1)

# Department accuracy
ax5 = fig.add_subplot(2, 3, 5)
dept_acc = df.groupby('department')['is_correct'].mean().sort_values(ascending=True)
dept_acc.plot(kind='barh', ax=ax5, color='mediumpurple', edgecolor='black')
ax5.set_title('Accuracy by Department')
ax5.set_xlim(0, 1)

# Winner accuracy
ax6 = fig.add_subplot(2, 3, 6)
winner_acc = df.groupby('winner')['is_correct'].mean()
winner_acc.plot(kind='bar', ax=ax6, color=['#3498db', '#e74c3c'], edgecolor='black')
ax6.set_title('Accuracy by Debate Winner')
ax6.set_ylim(0, 1)

plt.suptitle('Medical Error Detection - Demographic Analysis Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f"{OUTPUT_DIR}/summary_dashboard.png", dpi=150)
plt.show()

## 14. Export Results

In [ ]:
# Export full dataset
df.to_csv(f"{OUTPUT_DIR}/demographic_analysis_full.csv", index=False)
print(f"Full dataset exported to: {OUTPUT_DIR}/demographic_analysis_full.csv")

# Export summary metrics
summary_data = {
    'overall': {
        'total_cases': len(df),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc
    },
    'by_age_group': age_metrics_df.to_dict('records'),
    'by_gender': gender_metrics_df.to_dict('records'),
    'by_error_type': error_metrics_df.to_dict('records'),
    'by_disease_type': disease_metrics_df.to_dict('records'),
    'by_department': dept_metrics_df.to_dict('records'),
    'by_winner': winner_metrics_df.to_dict('records')
}

with open(f"{OUTPUT_DIR}/demographic_analysis_summary.json", 'w') as f:
    json.dump(summary_data, f, indent=2, default=str)
print(f"Summary exported to: {OUTPUT_DIR}/demographic_analysis_summary.json")

In [ ]:
# Print final summary
print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print(f"\nFiles saved to: {OUTPUT_DIR}/")
print("\nGenerated files:")
for f in Path(OUTPUT_DIR).glob('*'):
    print(f"  - {f.name}")